# Example: Stress-Testing the Minimum-Variance Portfolio

In this example, we run systematic "what-if" experiments on the baseline minimum-variance portfolio from the previous example. We shift correlations, simulate price drops, increase trading costs, and use Monte Carlo simulation to produce confidence bands around our performance metrics. Let's dive in!

> __Learning Objectives:__
>
> * __Correlation and Tail-Risk Stress Testing:__ Implement the crisis-regime blending formula to shift correlations toward perfect co-movement, visualize the eigenvalue collapse, and quantify concentration-driven losses from price drops. Observe how the optimizer's ability to diversify degrades as the correlation structure becomes more uniform.
> * __Monte Carlo Confidence Bands:__ Generate 1,000 simulated return paths and compute a distribution of portfolio outcomes (wealth, drawdown, Sharpe) with 68%, 95%, and 99% confidence intervals. Use these bands to transform single-point performance estimates into full probability distributions.
> * __Baseline Scorecard:__ Assemble a Min-Variance vs. Equal-Weight comparison scorecard with both point estimates and Monte Carlo percentiles. This scorecard serves as the quantitative benchmark that the AI rebalancing engine in Sessions 2 through 4 must improve upon.

## Setup, Data and Prerequisites
Load our packages and the baseline portfolio from the previous example.

In [ ]:
# --- Load packages and helper functions from Include.jl ---
include("Include.jl");

We use the [JLD2 `load(...)` function](https://juliaio.github.io/JLD2.jl/stable/) to read the baseline portfolio and synthetic return data generated in the previous example. The portfolio weights are stored in `baseline_weights::Vector{Float64}`, the estimated parameters in `μ_hat::Vector{Float64}` and `Σ_hat::Matrix{Float64}`, and the return data in `df_returns::DataFrame`.

In [ ]:
let
    # --- Step 1: Load baseline portfolio from the previous example ---
    portfolio_data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    returns_data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));

    # --- Step 2: Unpack into global-scope constants for use in later cells ---
    global baseline_weights = portfolio_data["weights"];       # optimal weights from min-variance solve
    global μ_hat = portfolio_data["mu_hat"];                   # estimated expected return vector
    global Σ_hat = portfolio_data["Sigma_hat"];                # estimated covariance matrix
    global asset_names = portfolio_data["asset_names"];        # string names for each asset
    global df_returns = returns_data["returns"];               # T × N DataFrame of synthetic returns
    global C_true = returns_data["C_true"];                    # ground-truth correlation matrix
    global N = length(asset_names);                            # number of assets
    global R_target = 0.05 / 252;                              # 5% annual target, converted to daily
    global bounds = hcat(zeros(N), ones(N));                   # long-only constraints: 0 ≤ wᵢ ≤ 1

    # --- Step 3: Display summary ---
    println("Loaded baseline portfolio with $(N) assets: $(asset_names)")
    println("Baseline weights: $(round.(baseline_weights .* 100, digits=1))%")
end

___
## Task 1: Stress Test, Correlation Shifts
We perturb the correlation structure toward a "crisis regime" where all assets become more correlated. Using the blending formula $\tilde{\mathbf{C}} = (1-\alpha)\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}$, we test $\alpha \in \{0.0, 0.10, 0.25, 0.50\}$ and re-solve the optimization under each stressed covariance matrix.

> __What are we going to do?__
>
> For each stress level $\alpha$, we blend the estimated correlation matrix toward the all-ones matrix (perfect correlation), reconstruct $\boldsymbol{\Sigma}$, solve the quadratic program, and record the new weights and portfolio volatility. Then we visualize the weight migration and eigenvalue collapse.

The results should show that as correlations increase, the diversification benefit shrinks. Portfolio variance _increases_ even though the optimizer is doing its best. The Bond becomes even more dominant because it is the only asset offering genuine diversification.

We use the [`build(...)`](../../code/docs/build/session1.html) and [`solve_minvariance(...)`](../../code/docs/build/session1.html) functions to re-solve the portfolio problem at each stress level, and the [`compute_turnover(...)`](../../code/docs/build/session1.html) function to measure how much the weights shift. The results are stored in the `stress_results::DataFrame` variable.

In [ ]:
let
    # --- Step 1: Extract estimated standard deviations and correlation from Σ_hat ---
    σ_hat = sqrt.(diag(Σ_hat));          # per-asset daily volatilities
    D_inv = diagm(1.0 ./ σ_hat);         # inverse of diagonal volatility matrix
    C_hat = D_inv * Σ_hat * D_inv;       # estimated correlation matrix: C = D⁻¹ Σ D⁻¹
    D_mat = diagm(σ_hat);                # diagonal volatility matrix for reconstruction

    # --- Step 2: Define stress levels and the all-ones correlation target ---
    αs = [0.0, 0.10, 0.25, 0.50];       # blending parameters (0 = baseline, 1 = perfect correlation)
    ones_matrix = ones(N, N);            # target: all pairwise correlations = 1

    # --- Step 3: Loop over stress levels, solve min-variance at each ---
    stress_results = DataFrame();

    for α in αs
        
        # Blend correlation toward crisis: C̃ = (1−α)C + α·11ᵀ
        C_stressed = (1 - α) .* C_hat .+ α .* ones_matrix;

        # Rebuild covariance matrix: Σ̃ = D·C̃·D
        Σ_stressed = D_mat * C_stressed * D_mat;

        # Solve the min-variance problem with stressed covariance
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_hat, Σ = Σ_stressed, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # Compute annualized portfolio volatility and turnover vs. baseline
        vol_annual = round(sqrt(result.variance) * sqrt(252) * 100, digits=2);
        turnover = round(compute_turnover(baseline_weights, result.weights), digits=4);

        # Store one row per stress level
        row = DataFrame(
            "α" => α,
            [name => round(w * 100, digits=1) for (name, w) in zip(asset_names, result.weights)]...,
            "Vol (%)" => vol_annual,
            "Turnover" => turnover
        );
        stress_results = vcat(stress_results, row);
    end

    # --- Step 4: Display the stress test results ---
    println("Correlation Stress Test Results:")
    pretty_table(stress_results, tf=tf_markdown)
end

**Visualize:** The following code compares the baseline ($\alpha = 0$) and stressed ($\alpha = 0.5$) correlation matrices side by side, and examines how the eigenvalue spectrum collapses under stress.

In [ ]:
let
    # --- Step 1: Compute baseline estimated correlation matrix ---
    σ_hat = sqrt.(diag(Σ_hat));
    D_inv = diagm(1.0 ./ σ_hat);
    C_hat = D_inv * Σ_hat * D_inv;       # estimated correlation: C = D⁻¹ Σ D⁻¹
    D_mat = diagm(σ_hat);
    ones_matrix = ones(N, N);
    
    # --- Step 2: Compute stressed correlation at α = 0.5 ---
    α_stress = 0.5;
    C_stressed = (1 - α_stress) .* C_hat .+ α_stress .* ones_matrix;
    
    # --- Step 3: Plot baseline vs stressed correlation heatmaps ---
    p1 = heatmap(C_hat, title="Baseline (α=0)", xticks=(1:N, asset_names),
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p2 = heatmap(C_stressed, title="Stressed (α=0.5)", xticks=(1:N, asset_names),
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    
    # --- Step 4: Eigenvalue comparison across multiple stress levels ---
    αs_fine = [0.0, 0.1, 0.25, 0.5, 0.75];
    p3 = plot(xlabel="Principal Component", ylabel="Eigenvalue (×10⁴)",
        title="Eigenvalue Collapse Under Stress", legend=:topright);
    for α ∈ αs_fine
        C_s = (1 - α) .* C_hat .+ α .* ones_matrix;     # blend correlation
        Σ_s = D_mat * C_s * D_mat;                        # reconstruct covariance
        λ_s = eigvals(Σ_s) |> sort |> reverse;            # eigenvalues, largest first
        plot!(p3, 1:N, λ_s .* 10000, marker=:circle, ms=4, label="α=$(α)", lw=2);
    end
    
    # --- Step 5: Combine into a single layout ---
    plot(p1, p2, p3, layout=@layout([a b; c{0.6h}]), size=(900, 700))
end

___
## Task 2: Stress Test, Price Drops and Monte Carlo Simulation
We first compute the deterministic impact of sudden price drops, then use Monte Carlo simulation to generate 1,000 return paths and visualize the _distribution_ of portfolio outcomes with confidence bands.

> __What are we going to do?__
>
> Two parts: (1) compute the portfolio-level loss from individual asset drops (deterministic), and (2) simulate 1,000 return paths from $\mathcal{N}(\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$, compute cumulative wealth for each path, and plot the fan chart with 68%, 95%, and 99% confidence bands.

The heatmap will reveal concentration risk, while the Monte Carlo fan chart shows how _uncertain_ the portfolio's future is. The confidence bands widen over time, and the drawdown distribution gives us a probabilistic view of worst-case losses.

The following code computes the portfolio-level loss for each asset at each drop level. The results are stored in the `drop_results::DataFrame` variable.

In [ ]:
let
    # --- Step 1: Define drop scenarios (10%, 20%, 40% price declines) ---
    drop_levels = [-0.10, -0.20, -0.40];
    
    # --- Step 2: Compute portfolio-level loss for each asset × drop combination ---
    drop_results = DataFrame();

    for i in 1:N
        for drop in drop_levels
            
            # Portfolio loss = weight_i × drop (linear approximation)
            portfolio_loss = baseline_weights[i] * drop * 100;

            row = DataFrame(
                "Asset" => asset_names[i],
                "Weight (%)" => round(baseline_weights[i] * 100, digits=1),
                "Drop (%)" => Int(drop * 100),
                "Portfolio Loss (%)" => round(portfolio_loss, digits=2)
            );
            drop_results = vcat(drop_results, row);
        end
    end

    # --- Step 3: Display the price drop stress test table ---
    println("Price Drop Stress Test (portfolio-level impact):")
    pretty_table(drop_results, tf=tf_markdown)
end

**Visualize:** Heatmap of portfolio losses across assets and drop levels.

> __What should you see?__
>
> A heatmap where the color intensity tracks the portfolio-level loss. The row corresponding to the highest-weight asset will show the deepest red, indicating where the concentration risk lives.

The following code builds the loss matrix and renders the heatmap.

In [ ]:
let
    # --- Step 1: Define drop levels (positive for loss computation) ---
    drop_levels = [0.10, 0.20, 0.40];

    # --- Step 2: Build loss matrix (rows = assets, cols = drop levels) ---
    loss_matrix = zeros(N, length(drop_levels));
    for (j, drop) in enumerate(drop_levels)
        for i in 1:N
            loss_matrix[i, j] = baseline_weights[i] * drop * 100;  # portfolio loss as %
        end
    end

    # --- Step 3: Render heatmap ---
    heatmap(["−10%", "−20%", "−40%"], asset_names, loss_matrix,
        xlabel="Price Drop", ylabel="Asset", title="Portfolio Loss (%) by Asset Drop",
        color=:Reds, size=(600, 400), clims=(0, maximum(loss_matrix)))
end

**Monte Carlo Simulation:** Now we simulate 1,000 return paths of length $T = 252$ days (1 year forward) from the estimated distribution $\mathcal{N}(\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$. For each path, we compute the cumulative wealth of the min-variance portfolio and an equal-weight benchmark.

> __What should you see?__
>
> A fan chart with gray individual paths, colored confidence bands (68%, 95%, 99%), and the expected wealth trajectory. The width of the bands at each time step quantifies the portfolio's uncertainty. Compare the min-variance bands to the equal-weight bands.

The following code draws from a [`MvNormal`](https://juliastats.org/Distributions.jl/stable/multivariate/#Distributions.MvNormal) distribution and computes cumulative wealth for both portfolios. The results are stored in `mc_mv_wealth::Matrix{Float64}` and `mc_ew_wealth::Matrix{Float64}`.

In [ ]:
let
    # --- Step 1: Set Monte Carlo parameters ---
    n_trials = 1000;      # number of simulated return paths
    T_sim = 252;           # simulation horizon: 1 year of trading days
    
    # --- Step 2: Build the multivariate normal return distribution ---
    dist = MvNormal(μ_hat, Σ_hat);
    
    # --- Step 3: Initialize wealth arrays for min-variance and equal-weight ---
    w_equal = fill(1.0/N, N);                          # equal-weight benchmark
    mv_wealth = zeros(T_sim + 1, n_trials);             # (T+1) × n_trials matrix
    ew_wealth = zeros(T_sim + 1, n_trials);
    mv_wealth[1, :] .= 1.0;                            # start with $1
    ew_wealth[1, :] .= 1.0;
    
    # --- Step 4: Simulate wealth paths ---
    for trial ∈ 1:n_trials
        for t ∈ 1:T_sim
            r_t = rand(dist);  # draw N-vector of daily returns
            mv_wealth[t+1, trial] = mv_wealth[t, trial] * (1.0 + dot(baseline_weights, r_t));
            ew_wealth[t+1, trial] = ew_wealth[t, trial] * (1.0 + dot(w_equal, r_t));
        end
    end
    
    # --- Step 5: Save for use in the scorecard cell ---
    global mc_mv_wealth = mv_wealth;
    global mc_ew_wealth = ew_wealth;
    
    # --- Step 6: Compute per-time-step statistics for the fan chart ---
    mv_mean = vec(mean(mv_wealth, dims=2));   # expected wealth at each day
    mv_std = vec(std(mv_wealth, dims=2));     # standard deviation at each day
    
    # --- Step 7: Plot the fan chart ---
    days = 0:T_sim;
    p = plot(title="Monte Carlo: Min-Variance Portfolio ($(n_trials) paths)",
        xlabel="Trading Day", ylabel="Cumulative Wealth (\$1 initial)",
        legend=:topleft, size=(750, 450));
    
    # Subset of individual paths for visual context
    for i ∈ 1:min(200, n_trials)
        plot!(p, days, mv_wealth[:, i], color=:gray86, lw=0.5, alpha=0.3, label="");
    end
    
    # Confidence bands: 99%, 95%, 68% (widest to narrowest)
    plot!(p, days, mv_mean .+ 2.576 .* mv_std, fillrange=mv_mean .- 2.576 .* mv_std,
        fillalpha=0.15, color=:steelblue, lw=0, label="99% CI");
    plot!(p, days, mv_mean .+ 1.96 .* mv_std, fillrange=mv_mean .- 1.96 .* mv_std,
        fillalpha=0.2, color=:steelblue, lw=0, label="95% CI");
    plot!(p, days, mv_mean .+ mv_std, fillrange=mv_mean .- mv_std,
        fillalpha=0.3, color=:steelblue, lw=0, label="68% CI");
    
    # Expected value trajectory
    plot!(p, days, mv_mean, lw=2.5, color=:darkblue, label="E[Wealth]");
    
    p
end

**Distribution of Outcomes:** Next we examine the terminal wealth and maximum drawdown distributions across the 1,000 simulated paths, comparing the min-variance portfolio to the equal-weight benchmark.

> __What should you see?__
>
> The min-variance portfolio should have a _tighter_ terminal wealth distribution (lower variance) but a similar or lower median. The drawdown distributions reveal the tail risk, showing the worst-case losses across the simulated paths.

The following code computes terminal wealth and maximum drawdown for each simulated path, then renders side-by-side histograms.

In [ ]:
let
    n_trials = size(mc_mv_wealth, 2);
    T_sim = size(mc_mv_wealth, 1) - 1;
    
    # --- Step 1: Extract terminal wealth from the last row of each path ---
    mv_terminal = mc_mv_wealth[end, :];
    ew_terminal = mc_ew_wealth[end, :];
    
    # --- Step 2: Compute maximum drawdown for each simulated path ---
    mv_drawdowns = zeros(n_trials);
    ew_drawdowns = zeros(n_trials);
    for i ∈ 1:n_trials
        # Running peak and drawdown for min-variance
        mv_peak = accumulate(max, mc_mv_wealth[:, i]);
        mv_dd = (mv_peak .- mc_mv_wealth[:, i]) ./ mv_peak;
        mv_drawdowns[i] = maximum(mv_dd);
        
        # Running peak and drawdown for equal-weight
        ew_peak = accumulate(max, mc_ew_wealth[:, i]);
        ew_dd = (ew_peak .- mc_ew_wealth[:, i]) ./ ew_peak;
        ew_drawdowns[i] = maximum(ew_dd);
    end
    
    # --- Step 3: Plot terminal wealth histograms with median markers ---
    p1 = histogram(mv_terminal, bins=50, alpha=0.6, label="Min-Variance", color=:steelblue,
        xlabel="Terminal Wealth (\$1 initial)", ylabel="Frequency", title="Terminal Wealth Distribution");
    histogram!(p1, ew_terminal, bins=50, alpha=0.5, label="Equal-Weight", color=:coral);
    vline!(p1, [median(mv_terminal)], lw=2, ls=:dash, color=:darkblue, label="MV median");
    vline!(p1, [median(ew_terminal)], lw=2, ls=:dash, color=:darkred, label="EW median");
    
    # --- Step 4: Plot max drawdown histograms ---
    p2 = histogram(mv_drawdowns .* 100, bins=50, alpha=0.6, label="Min-Variance", color=:steelblue,
        xlabel="Max Drawdown (%)", ylabel="Frequency", title="Max Drawdown Distribution");
    histogram!(p2, ew_drawdowns .* 100, bins=50, alpha=0.5, label="Equal-Weight", color=:coral);
    
    # --- Step 5: Combine into side-by-side layout ---
    plot(p1, p2, layout=(1,2), size=(1000, 400), legend=:topright)
end

___
## Task 3: Trading Cost Stress Test and Baseline Scorecard
We evaluate how portfolio performance changes as trading costs increase, then assemble a scorecard that summarizes the baseline min-variance portfolio's performance, including Monte Carlo confidence intervals.

> __What are we going to do?__
>
> First, we compute trading costs at 1x, 2x, 5x, and 10x the baseline assumption. Then we build the full scorecard with both point estimates (from the synthetic data) and Monte Carlo percentiles (from the 1,000 simulated paths). This scorecard is the quantitative benchmark that the AI rebalancing engine in Session 2 needs to beat.

High-turnover portfolios suffer most from cost increases. The Monte Carlo percentiles add uncertainty quantification to every metric, giving us a range instead of just a point estimate.

The following code uses the [`compute_turnover(...)`](../../code/docs/build/session1.html) function to measure the rebalance from equal-weight to min-variance, then scales the cost at each multiplier level. The results are stored in the `cost_results::DataFrame` variable.

In [ ]:
let
    # --- Step 1: Define baseline cost assumption and multiplier levels ---
    base_cost_bps = 10.0;                   # baseline: 10 bps per unit traded
    cost_multipliers = [1, 2, 5, 10];       # stress levels: 1x through 10x

    # --- Step 2: Compute turnover from equal-weight to min-variance ---
    w_equal = fill(1.0 / N, N);
    turnover = compute_turnover(w_equal, baseline_weights);

    # --- Step 3: Compute trading cost at each multiplier level ---
    cost_results = DataFrame();

    for mult in cost_multipliers
        cost_bps = base_cost_bps * mult;                      # scaled cost per unit traded
        trading_cost = turnover * (cost_bps / 10000) * 100;   # total cost as % of portfolio

        row = DataFrame(
            "Cost Multiplier" => "$(mult)×",
            "Cost (bps/trade)" => Int(cost_bps),
            "Turnover" => round(turnover, digits=4),
            "Trading Cost (%)" => round(trading_cost, digits=4)
        );
        cost_results = vcat(cost_results, row);
    end

    # --- Step 4: Display results ---
    println("Trading Cost Stress Test (equal-weight → min-variance rebalance):")
    pretty_table(cost_results, tf=tf_markdown)
end

**Baseline Scorecard:** Now we assemble the scorecard, combining point estimates from the historical synthetic data with Monte Carlo percentiles from the simulated paths.

> __What should you see?__
>
> A comparison table showing Min-Variance vs. Equal-Weight across all metrics. The Monte Carlo columns show the 5th and 95th percentile outcomes, representing the "bad case" and "good case" scenarios.

The following code uses the [`compute_drawdown(...)`](../../code/docs/build/session1.html) function to measure historical drawdown, then combines it with Monte Carlo percentiles into the `scorecard::DataFrame` variable.

In [ ]:
let
    # --- Step 1: Point estimates from synthetic data ---
    R = Matrix(df_returns);                          # T × N return matrix
    portfolio_returns = R * baseline_weights;         # min-variance portfolio daily returns
    w_equal = fill(1.0 / N, N);
    eq_returns = R * w_equal;                        # equal-weight portfolio daily returns
    
    # --- Step 2: Compute min-variance performance metrics ---
    mv_ret = mean(portfolio_returns) * 252 * 100;                # annualized return (%)
    mv_vol = std(portfolio_returns) * sqrt(252) * 100;           # annualized volatility (%)
    mv_sharpe = mv_ret / mv_vol;                                 # Sharpe ratio
    mv_dd = compute_drawdown(portfolio_returns) * 100;           # max drawdown (%)
    mv_turnover = compute_turnover(w_equal, baseline_weights);   # turnover vs equal-weight
    mv_cost = mv_turnover * (10.0 / 10000) * 100;               # trading cost at 10 bps (%)
    
    # --- Step 3: Compute equal-weight performance metrics ---
    ew_ret = mean(eq_returns) * 252 * 100;
    ew_vol = std(eq_returns) * sqrt(252) * 100;
    ew_sharpe = ew_ret / ew_vol;
    ew_dd = compute_drawdown(eq_returns) * 100;
    
    # --- Step 4: Monte Carlo percentiles from simulated paths ---
    n_trials = size(mc_mv_wealth, 2);
    mv_terminal = mc_mv_wealth[end, :];
    ew_terminal = mc_ew_wealth[end, :];
    
    # Max drawdown per simulated path
    mv_mc_dd = zeros(n_trials);
    ew_mc_dd = zeros(n_trials);
    for i ∈ 1:n_trials
        pk = accumulate(max, mc_mv_wealth[:, i]);
        mv_mc_dd[i] = maximum((pk .- mc_mv_wealth[:, i]) ./ pk) * 100;
        pk_e = accumulate(max, mc_ew_wealth[:, i]);
        ew_mc_dd[i] = maximum((pk_e .- mc_ew_wealth[:, i]) ./ pk_e) * 100;
    end
    
    # Annualized return from terminal wealth (1-year horizon, so exponent = 1)
    mv_mc_ret = (mv_terminal .^ (252/252) .- 1.0) .* 100;
    ew_mc_ret = (ew_terminal .^ (252/252) .- 1.0) .* 100;
    
    # --- Step 5: Build the scorecard DataFrame ---
    scorecard = DataFrame(
        "Metric" => [
            "Expected Return (annual %)",
            "Volatility (annual %)",
            "Sharpe Ratio",
            "Max Drawdown (%)",
            "Turnover (vs equal-wt)",
            "Trading Cost (%, 10 bps)",
            "MC Return, 5th pctile (%)",
            "MC Return, 95th pctile (%)",
            "MC Max DD, 95th pctile (%)"
        ],
        "Min-Variance" => [
            round(mv_ret, digits=2),
            round(mv_vol, digits=2),
            round(mv_sharpe, digits=2),
            round(mv_dd, digits=2),
            round(mv_turnover, digits=4),
            round(mv_cost, digits=4),
            round(quantile(mv_mc_ret, 0.05), digits=2),
            round(quantile(mv_mc_ret, 0.95), digits=2),
            round(quantile(mv_mc_dd, 0.95), digits=2)
        ],
        "Equal-Weight" => [
            round(ew_ret, digits=2),
            round(ew_vol, digits=2),
            round(ew_sharpe, digits=2),
            round(ew_dd, digits=2),
            0.0,
            0.0,
            round(quantile(ew_mc_ret, 0.05), digits=2),
            round(quantile(ew_mc_ret, 0.95), digits=2),
            round(quantile(ew_mc_dd, 0.95), digits=2)
        ]
    );
    
    # --- Step 6: Display the scorecard ---
    println("="^65)
    println("  BASELINE SCORECARD: Min-Variance vs. Equal-Weight")
    println("="^65)
    pretty_table(scorecard, tf = tf_markdown)
    
    # --- Step 7: Save scorecard and data for later sessions ---
    save(joinpath(_PATH_TO_DATA, "baseline-scorecard.jld2"),
        Dict("scorecard" => scorecard, "portfolio_returns" => portfolio_returns,
             "baseline_weights" => baseline_weights,
             "mc_mv_wealth" => mc_mv_wealth, "mc_ew_wealth" => mc_ew_wealth));
end

**Visualize:** Cumulative wealth comparison using the _historical_ synthetic data, plus a grouped bar chart comparing Min-Variance vs. Equal-Weight across the key scorecard metrics.

> __What should you see?__
>
> Two wealth curves starting at \$1.00. The min-variance portfolio should show lower volatility (smoother path) but potentially lower total return. The bar chart makes the trade-offs immediately visible.

The following code computes cumulative wealth from the synthetic return data using the [`compute_drawdown(...)`](../../code/docs/build/session1.html) function and renders the comparison plots.

In [ ]:
let
    # --- Step 1: Compute cumulative wealth for both portfolios ---
    R = Matrix(df_returns);
    w_equal = fill(1.0 / N, N);
    
    mv_returns = R * baseline_weights;               # daily returns: min-variance
    mv_wealth = cumprod(1.0 .+ mv_returns);          # cumulative wealth from $1
    
    ew_returns = R * w_equal;                        # daily returns: equal-weight
    ew_wealth = cumprod(1.0 .+ ew_returns);

    T_hist = length(mv_wealth);
    days = 1:T_hist;

    # --- Step 2: Cumulative wealth plot ---
    p1 = plot(days, mv_wealth, label="Min-Variance", lw=2, color=:steelblue,
        xlabel="Trading Day", ylabel="Cumulative Wealth (\$1 initial)",
        title="Historical Synthetic Data", legend=:topleft);
    plot!(p1, days, ew_wealth, label="Equal-Weight", lw=2, color=:coral, linestyle=:dash);
    
    # --- Step 3: Compute scorecard metrics for the bar chart ---
    mv_ret = mean(mv_returns) * 252 * 100;           # annualized return (%)
    ew_ret = mean(ew_returns) * 252 * 100;
    mv_vol = std(mv_returns) * sqrt(252) * 100;      # annualized volatility (%)
    ew_vol = std(ew_returns) * sqrt(252) * 100;
    mv_dd = compute_drawdown(mv_returns) * 100;      # max drawdown (%)
    ew_dd = compute_drawdown(ew_returns) * 100;
    
    # --- Step 4: Grouped bar chart comparing key metrics ---
    metrics = ["Return (%)", "Volatility (%)", "Max DD (%)"];
    mv_vals = [mv_ret, mv_vol, mv_dd];
    ew_vals = [ew_ret, ew_vol, ew_dd];
    
    p2 = groupedbar(metrics, hcat(mv_vals, ew_vals),
        bar_position=:dodge, label=["Min-Variance" "Equal-Weight"],
        color=[:steelblue :coral], ylabel="Value",
        title="Scorecard Comparison", legend=:topright);
    
    # --- Step 5: Combine into side-by-side layout ---
    plot(p1, p2, layout=(1,2), size=(1100, 400))
end

___
## Summary
This example stress-tested the baseline minimum-variance portfolio under correlation shifts, price drops, trading cost increases, and Monte Carlo simulation to quantify the range of possible outcomes. The resulting scorecard provides the quantitative benchmark that the AI rebalancing engine in Session 2 must improve upon.

> __Key Takeaways:__
>
> * **Correlation stress collapses the eigenstructure:** As the blending parameter increases, the eigenvalue spectrum concentrates into a single dominant component, leaving the optimizer fewer dimensions to diversify across. The Bond asset dominates even more because it is the only asset offering genuine decorrelation.
> * **Monte Carlo transforms point estimates into distributions:** The 1,000 simulated paths provide 5th and 95th percentile bounds on return, drawdown, and terminal wealth. These bounds reveal the full range of outcomes the min-variance portfolio can produce over a one-year horizon.
> * **The baseline scorecard enables benchmarking:** The Min-Variance vs. Equal-Weight comparison, with both point estimates and Monte Carlo percentiles, gives a complete performance summary. This scorecard is saved to the data directory for use in subsequent sessions.

The scorecard, Monte Carlo wealth paths, and portfolio data are saved to the `data/` directory for use in subsequent sessions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.